# 05 SequencerWidget Tutorial

Monophonic and polyphonic step sequencers using `SequencerWidget` and `NoteComposer`.

## Monophonic Sequencer

A single-voice sequencer is the simplest case: one `NoteComposer` (voice 0) holds all the steps.

In [1]:
from nbplay import SequencerWidget

# A monophonic sequencer with 8 steps (num_voices defaults to 1)
mono = SequencerWidget(length=8, bpm=120.0)

# Program an ascending arpeggio on voice 0 (the default)
for step, note in enumerate([60, 64, 67, 72, 67, 64, 60, 55]):
    mono.set_step(step, note=note, velocity=100, active=True)

mono

## Reading Steps Back

`steps` is a shortcut for `voices[0].steps` — both return the same list of step dicts.

In [2]:
# steps property aliases voice 0
print("step 0 via .steps:", mono.steps[0])
print("step 0 via .voices:", mono.voices[0].steps[0])
assert mono.steps == mono.voices[0].steps

step 0 via .steps: {'note': 60, 'velocity': 100, 'duration_ticks': 1, 'active': True}
step 0 via .voices: {'note': 60, 'velocity': 100, 'duration_ticks': 1, 'active': True}


## Polyphonic Sequencer

Set `num_voices` to create multiple independent voices. Each voice is a `NoteComposer`
that holds its own set of steps. The browser-side scheduler plays **all active voices**
simultaneously on each step.

In [3]:
import ipywidgets as widgets

from nbplay import SequencerWidget

# Three-voice polyphonic sequencer: root, third, fifth
poly = SequencerWidget(length=8, bpm=118.0, num_voices=3)

# Voice 0 — root notes (C)
for step, note in enumerate([60, 60, 60, 60, 60, 60, 60, 60]):
    poly.set_step(step, note=note, velocity=100, active=True, voice=0)

# Voice 1 — major thirds (E)
for step, note in enumerate([64, 64, 64, 64, 64, 64, 64, 64]):
    poly.set_step(step, note=note, velocity=85, active=True, voice=1)

# Voice 2 — fifths (G), only on beats 0, 2, 4, 6
for step in [0, 2, 4, 6]:
    poly.set_step(step, note=67, velocity=90, active=True, voice=2)

poly

## Working with NoteComposer Objects

Each voice is a `NoteComposer` instance accessible via the `composers` or `voices` property.
You can manipulate them directly and the widget stays in sync.

In [4]:
from nbplay import NoteComposer

# Access the three composers
root_voice = poly.composers[0]
third_voice = poly.composers[1]
fifth_voice = poly.composers[2]

print(f"Number of voices: {poly.num_voices}")
print(f"Root voice active steps:  {sum(1 for s in root_voice.steps if s['active'])}")
print(f"Third voice active steps: {sum(1 for s in third_voice.steps if s['active'])}")
print(f"Fifth voice active steps: {sum(1 for s in fifth_voice.steps if s['active'])}")

# Toggle a step off on voice 1 (the third)
third_voice.toggle_step(3)
print(f"\nAfter toggling step 3 on third voice:")
print(f"  step 3 active = {third_voice.steps[3]['active']}")

Number of voices: 3
Root voice active steps:  8
Third voice active steps: 8
Fifth voice active steps: 4

After toggling step 3 on third voice:
  step 3 active = False


## Standalone NoteComposer

`NoteComposer` can be used outside a widget — useful for building patterns programmatically
before attaching them to a sequencer.

In [5]:
from nbplay import NoteComposer

# Build a bass pattern
bass_line = NoteComposer(length=8)
for step, (note, vel) in enumerate([
    (36, 120), (36, 60), (38, 90), (36, 60),
    (41, 110), (36, 60), (43, 95), (36, 60),
]):
    bass_line.set_step(step, note=note, velocity=vel, active=True)

# Convert to a Rust Pattern object
pattern = bass_line.to_pattern()
print(bass_line)
print(pattern)

NoteComposer(length=8, active=8)
Pattern(length=8, active_steps=8, loop=true)


## Multiple Monophonic Sequencers

For independent instrument parts (lead, bass, drums), use separate monophonic sequencers
— each routes to its own mixer channel in a Session.

In [6]:
import ipywidgets as widgets

from nbplay import SequencerWidget

lead_seq = SequencerWidget(length=8, bpm=118.0)
for step, note in enumerate([72, 76, 79, 83, 79, 76, 74, 71]):
    lead_seq.set_step(step, note=note, velocity=102, active=True)

bass_seq = SequencerWidget(length=8, bpm=118.0)
for step, velocity in enumerate([118, 64, 88, 64, 110, 64, 92, 64]):
    bass_seq.set_step(step, note=48, velocity=velocity, active=True)

drum_seq = SequencerWidget(length=8, bpm=118.0)
for step in [0, 3, 4, 7]:
    drum_seq.set_step(step, note=60, velocity=112, active=True)

widgets.HBox([lead_seq, bass_seq, drum_seq], layout=widgets.Layout(gap="16px"))

## Converting to Rust Patterns

`to_pattern()` on a sequencer converts voice 0 into a Rust `Pattern` object.
Each `NoteComposer` also has its own `to_pattern()` method.

In [7]:
# From the polyphonic sequencer — each voice converts independently
for i, composer in enumerate(poly.composers):
    pattern = composer.to_pattern()
    print(f"Voice {i}: {pattern}")

# From a monophonic sequencer (uses voice 0)
print(f"\nLead: {lead_seq.to_pattern()}")

Voice 0: Pattern(length=8, active_steps=8, loop=true)
Voice 1: Pattern(length=8, active_steps=7, loop=true)
Voice 2: Pattern(length=8, active_steps=4, loop=true)

Lead: Pattern(length=8, active_steps=8, loop=true)
